In [1]:
import pandas as pd

SUBJECTS = [
    "Maths",
    "Physics",
    "Chemistry",
    "English",
    "Computer_Science"
]

PASS_MARK = 33
MIN_ROWS_FOR_ML = 500


def load_data(filename):
    df = pd.read_csv(filename, dtype={"class": str})
    df["average_marks"] = df[SUBJECTS].mean(axis=1)

    status_list = []
    for i in range(len(df)):
        row_marks = df.iloc[i][SUBJECTS]
        if all(row_marks >= PASS_MARK):
            status_list.append("Pass")
        else:
            status_list.append("Fail")

    df["overall_status"] = status_list

    return df


def check_size(df):
    row_count = len(df)
    column_count = len(df.columns)
    meets_size_floor = row_count >= MIN_ROWS_FOR_ML

    result = {
        "row_count": row_count,
        "column_count": column_count,
        "meets_size_floor": meets_size_floor
    }

    return result


def check_label_balance(df):
    pass_count = 0
    fail_count = 0

    for status in df["overall_status"]:
        if status == "Pass":
            pass_count += 1
        else:
            fail_count += 1

    total = len(df)
    pass_pct = round(pass_count / total * 100, 2)
    fail_pct = round(fail_count / total * 100, 2)

    minority_pct = min(pass_pct, fail_pct)
    is_balanced = minority_pct >= 15

    result = {
        "pass_count": pass_count,
        "fail_count": fail_count,
        "pass_pct": pass_pct,
        "fail_pct": fail_pct,
        "is_balanced": is_balanced
    }

    return result


def check_label_circularity(df):
    correct = 0

    for i in range(len(df)):
        row_marks = df.iloc[i][SUBJECTS]

        if all(row_marks >= PASS_MARK):
            predicted = "Pass"
        else:
            predicted = "Fail"

        actual = df.iloc[i]["overall_status"]

        if predicted == actual:
            correct += 1

    accuracy = round(correct / len(df) * 100, 2)
    is_deterministic = accuracy == 100.0

    result = {
        "rule_accuracy_pct": accuracy,
        "label_is_deterministic": is_deterministic
    }

    return result


def check_signal_strength(df):
    correlation = df["attendance_pct"].corr(df["average_marks"])
    r_squared = correlation ** 2
    variance_explained_pct = round(r_squared * 100, 1)

    result = {
        "attendance_correlation": round(correlation, 3),
        "variance_explained_pct": variance_explained_pct
    }

    return result


def write_report(size_check, label_check, circularity_check, signal_check, filename):

    file = open(filename, "w")

    file.write("ML FEASIBILITY ASSESSMENT\n")
    file.write("=" * 45 + "\n\n")

    file.write("Business question: Should we build an ML model to predict\n")
    file.write("whether a student will Pass or Fail?\n\n")

    file.write("1. DATASET SIZE\n")
    file.write("-" * 30 + "\n")
    file.write(f"Rows: {size_check['row_count']}\n")
    file.write(f"Columns: {size_check['column_count']}\n")
    file.write(f"Minimum rows usually needed: {MIN_ROWS_FOR_ML}\n")
    file.write(f"Meets size requirement: {size_check['meets_size_floor']}\n\n")

    file.write("2. LABEL BALANCE\n")
    file.write("-" * 30 + "\n")
    file.write(f"Pass: {label_check['pass_count']} students ({label_check['pass_pct']}%)\n")
    file.write(f"Fail: {label_check['fail_count']} students ({label_check['fail_pct']}%)\n")
    file.write(f"Balanced enough: {label_check['is_balanced']}\n\n")

    file.write("3. LABEL CIRCULARITY CHECK\n")
    file.write("-" * 30 + "\n")
    file.write(f"A simple rule (any subject below {PASS_MARK} = Fail) matches\n")
    file.write(f"the real label {circularity_check['rule_accuracy_pct']}% of the time.\n\n")

    if circularity_check["label_is_deterministic"]:
        file.write("This means the label can already be worked out with a simple\n")
        file.write("rule. There is nothing extra for ML to learn here.\n\n")
    else:
        file.write("The label is not fully explained by a simple rule, so ML\n")
        file.write("might add real value here.\n\n")

    file.write("4. SIGNAL STRENGTH (Attendance)\n")
    file.write("-" * 30 + "\n")
    file.write(f"Attendance to marks correlation: {signal_check['attendance_correlation']}\n")
    file.write(f"Variance explained by attendance: {signal_check['variance_explained_pct']}%\n\n")

    file.write("5. FINAL VERDICT\n")
    file.write("-" * 30 + "\n")

    ml_justified = (
        size_check["meets_size_floor"]
        and not circularity_check["label_is_deterministic"]
        and signal_check["variance_explained_pct"] >= 15
    )

    if ml_justified:
        file.write("RECOMMENDATION: GO. ML is justified for this problem.\n")
        file.write("Suggested approach: Logistic Regression as a baseline model,\n")
        file.write("evaluated using Precision, Recall, and F1-score.\n")
    else:
        file.write("RECOMMENDATION: NO-GO. ML is not justified right now.\n\n")
        file.write("Reasons:\n")

        if not size_check["meets_size_floor"]:
            file.write(f"- Only {size_check['row_count']} rows available, below the {MIN_ROWS_FOR_ML} row guideline.\n")

        if circularity_check["label_is_deterministic"]:
            file.write("- The Pass/Fail label can be worked out perfectly with a simple rule,\n")
            file.write("  so there is nothing for a model to learn beyond that rule.\n")

        if signal_check["variance_explained_pct"] < 15:
            file.write("- Attendance alone does not explain enough of the variation\n")
            file.write("  in marks to be useful for prediction.\n")

        file.write("\nRecommended path instead:\n")
        file.write("- Keep using the simple rule-based system from Project 2.\n")
        file.write("  It is 100% accurate and easy to explain.\n")
        file.write("- Revisit ML only if new data is collected that exists BEFORE\n")
        file.write("  the final marks, such as quiz scores or assignment activity.\n")
        file.write(f"- Also revisit once the dataset grows past around {MIN_ROWS_FOR_ML} rows.\n")

    file.close()


def main():

    df = load_data("clean_student_marks.csv")

    size_check = check_size(df)
    label_check = check_label_balance(df)
    circularity_check = check_label_circularity(df)
    signal_check = check_signal_strength(df)

    write_report(size_check, label_check, circularity_check, signal_check, "feasibility_report.txt")

    print("ML feasibility check done!")
    print("Rows:", size_check["row_count"], "| Meets size floor:", size_check["meets_size_floor"])
    print("Rule accuracy:", circularity_check["rule_accuracy_pct"], "%")
    print("Attendance variance explained:", signal_check["variance_explained_pct"], "%")
    print("Report saved as feasibility_report.txt")


if __name__ == "__main__":
    main()

ML feasibility check done!
Rows: 117 | Meets size floor: False
Rule accuracy: 100.0 %
Attendance variance explained: 2.4 %
Report saved as feasibility_report.txt
